# L4d: Production-Planning Shortest Path

A production process with alternative routes can be modeled as a weighted directed graph. Each vertex represents a process state. Each directed edge represents an allowed production step. Its weight records the cost of that step. A feasible production plan is a route from the start state to completion. Choosing the least-cost plan is therefore a shortest-path problem.

> __Learning Objectives:__
>
> By the end of this lab, you should be able to:
> * __Represent the production process:__ Parse the process edge list into vertices, directed edges, and step costs. Identify the two candidate routes and use the nonnegative weights to justify Dijkstra's algorithm for this graph.
> * __Compute and validate the least-cost route:__ Calculate each candidate route's cost by hand, verify the result with Dijkstra and Bellman–Ford, and reconstruct the selected route from its predecessor map.
> * __Find the route-switching threshold:__ Change one step cost on a copy of the graph so the baseline remains available for comparison. Implement the break-even calculation that identifies when the discounted route becomes cheaper.

This lab applies that model to a nine-vertex process with two candidate routes. We calculate both route costs by hand and verify the cheaper route with the Dijkstra and Bellman–Ford algorithms developed in [L4c](../L4c/CHEME-5800-L4c-Lecture-ShortestPathAlgorithms-Fall-2026.ipynb). We then determine how far the cost of step `(3, 4)` must fall before the preferred route changes. Let's get started!
___

## Setup, Data, and Prerequisites
Run the next cell to evaluate [`Include.jl`](Include.jl) in the notebook's global scope with [the `include(...)` command](https://docs.julialang.org/en/v1/base/base/#include). The meeting-local setup file activates the pinned course environment and defines a path to the L4d data directory. It then loads the course package and student module and imports the packages used by the tables, figures, and checks.

In [ ]:
include(joinpath(@__DIR__, "Include.jl")); # activate the pinned environment and load the L4d dependencies

[The `VLDataScienceMachineLearningPackage.jl` course package](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/) provides the directed-graph model and shortest-path solver. [The `Test` standard library](https://docs.julialang.org/en/v1/stdlib/Test/) supplies executable checks. [The `DataFrames.jl` package](https://dataframes.juliadata.org/stable/) provides the table container used to assemble edge comparisons. [The `PrettyTables.jl` package](https://ronisbr.github.io/PrettyTables.jl/stable/) renders those records as readable tables. [The `Plots.jl` package](https://docs.juliaplots.org/stable/) draws the route figures.

The setup evaluates [`src/Compute.jl`](src/Compute.jl), which defines [the `L4dProductionPlanning` module](src/Compute.jl). Tasks 2 and 3 use its route-cost and route-reconstruction functions. Task 3 also asks you to complete its break-even function. The setup deliberately reloads this file each time it runs, so rerunning the setup cell makes saved edits available without restarting the kernel. Qualified calls such as [the `L4dProductionPlanning.route_cost(...)` function](src/Compute.jl) therefore resolve to the latest version of the module.

### Shared plotting setup
Tasks 2 and 3 use identical vertex positions for the baseline and discounted graphs. Holding the layout constant ensures that differences between the figures reflect only the changed edge cost and selected route. The `node_coordinates::Matrix{Float64}` array assigns one $(x,y)$ position to each vertex. The `start_vertex::Int64` and `finish_vertex::Int64` constants identify vertices 1 and 9 as the endpoints highlighted in both figures. These values control the display; the shortest-path solver reads topology and costs from the graph model:

In [ ]:
# Define the route-figure layout: one (x, y) row per vertex, matching the Task 1 schematic.
node_coordinates = [
    10.0 10.0 ; # 1 start
    11.0 10.0 ; # 2
    11.0 11.0 ; # 3 upper route
    13.0 11.0 ; # 4 upper route
    13.0 10.0 ; # 5
    11.0  9.0 ; # 6 lower route
    12.0  9.0 ; # 7 lower route
    13.0  9.0 ; # 8 lower route
    14.0 10.0 ; # 9 completion
];
start_vertex = 1;  # every candidate route begins at vertex 1
finish_vertex = 9; # every candidate route ends at vertex 9

[The `plotroute(...)` function](#Shared-plotting-setup) uses that fixed layout to draw every directed edge and its cost. It then redraws the selected route in red and marks the start and finish vertices. This common encoding makes a route change visible while preserving the surrounding network for comparison:

In [ ]:
"""
    plotroute(graphmodel::MySimpleDirectedGraphModel, route::Vector{Int64}, coordinates::Matrix{Float64};
        start::Int64 = start_vertex, finish::Int64 = finish_vertex)

Draw the graph on the given layout, label every edge with its cost, and highlight `route` in red.
Returns the current figure.
"""
function plotroute(graphmodel::MySimpleDirectedGraphModel, route::Vector{Int64}, coordinates::Matrix{Float64};
    start::Int64 = start_vertex, finish::Int64 = finish_vertex)

    # Initialize -
    route_edges = Set((route[i], route[i + 1]) for i in 1:(length(route) - 1));
    plot(); # start a fresh figure

    # Draw every edge in gray; label non-route edges once in black.
    for ((s, t), w) in graphmodel.edges
        xs = [coordinates[s, 1], coordinates[t, 1]];
        ys = [coordinates[s, 2], coordinates[t, 2]];
        plot!(xs, ys, arrow = true, color = :gray90, lw = 2, label = "")
        if !((s, t) in route_edges)
            annotate!(sum(xs) / 2, sum(ys) / 2 + 0.15, text(string(round(w, digits = 2)), 8, :black))
        end
    end

    # Redraw the selected route in red on top of the full graph.
    for (s, t) in route_edges
        xs = [coordinates[s, 1], coordinates[t, 1]];
        ys = [coordinates[s, 2], coordinates[t, 2]];
        plot!(xs, ys, arrow = true, color = :red, lw = 2, label = "")
        annotate!(sum(xs) / 2, sum(ys) / 2 + 0.20, text(string(round(graphmodel.edges[(s, t)], digits = 2)), 8, :red))
    end

    # Draw the numbered vertices: gray by default, green at the start, and red at the finish.
    scatter!(coordinates[:, 1], coordinates[:, 2], c = :gray, ms = 16, label = "")
    scatter!([coordinates[start, 1]], [coordinates[start, 2]], c = :green, ms = 16,
        markerstrokewidth = 2, markerstrokecolor = :darkgreen, label = "Start")
    scatter!([coordinates[finish, 1]], [coordinates[finish, 2]], c = :red, ms = 16,
        markerstrokewidth = 2, markerstrokecolor = :darkred, label = "Finish")
    for i in 1:size(coordinates, 1)
        color = (i == start || i == finish) ? :white : :black
        annotate!(coordinates[i, 1], coordinates[i, 2], text(string(i), 9, color))
    end

    plot!(axis = nothing, border = :none, legend = :outertopright, legendfontsize = 8,
        background_color = :white, xlim = (9.5, 14.5), ylim = (8.5, 11.5))
    return current()
end;

___

## Task 1: Build the production graph
In this task, we parse the production edge list and build the directed graph model used by the shortest-path solvers. We inspect the stored endpoint pairs, edge ids, and weights to verify both the input topology and Dijkstra's nonnegative-weight requirement before Task 2.

The process begins at vertex 1 and reaches a branch at vertex 2. The upper branch passes through vertices 3 and 4; the lower branch passes through vertices 6, 7, and 8. Both branches rejoin at vertex 5 before the process ends at vertex 9. The upper route uses five steps and includes branch costs of 4 and 8. The lower route uses six steps, but each of its four branch costs is 2. The smaller step count favors the upper route; the edge weights may favor the lower route. This contrast lets us test whether the weighted shortest-path solver minimizes edge count or accumulated cost.

<div>
    <center>
        <img src="figs/Fig-Branch-Schematic.svg" width="480" alt="Directed production graph: start vertex 1 leads to vertex 2, which splits into an upper route through vertices 3 and 4 and a lower route through vertices 6, 7, and 8; both routes rejoin at vertex 5 before the completion vertex 9"/>
    </center>
</div>

The process topology and costs are stored in [`data/Production-Process.edgelist`](data/Production-Process.edgelist), with one non-comment record per directed production step.

> __How is one production step encoded?__
>
> Each record has three comma-separated fields: `source`, `target`, and `cost`. The integer `source` and `target` fields identify the step's initial and final process states. The numeric `cost` field measures the expense of completing that step in arbitrary cost units. Lines beginning with `#` document the synthetic dataset and are not graph edges. This schema supplies exactly the two endpoints and one weight required to construct a weighted directed edge.

[The `MyGraphEdgeModels(...)` constructor](../../../code/src/Files.jl) reads the file and passes each non-comment record to the parser below. The parser removes surrounding whitespace, requires exactly three fields, converts the vertex identifiers to `Int64`, and converts the cost to `Float64`. Rejecting a malformed record here prevents an incomplete endpoint or cost from entering the graph model:

In [ ]:
"""
    edgerecordparser(record::String, delim::Char = ',') -> Tuple{Int64, Int64, Float64}

Parse one `source,target,cost` record into an `(Int64, Int64, Float64)` edge tuple.

### Arguments
- `record`: The edge record string to parse.
- `delim`: The delimiter used to split the record.

### Returns
- A tuple containing the source vertex id, target vertex id, and cost of the edge.

### Errors
- `ArgumentError`: The record does not have exactly three fields.
"""
function edgerecordparser(record::String, delim::Char = ',')

    fields = strip.(split(record, delim)); # remove whitespace around the fields
    length(fields) == 3 || throw(ArgumentError("expected source,target,cost but got: $(record)"))

    source = parse(Int64, fields[1]);  # source vertex id
    target = parse(Int64, fields[2]);  # target vertex id
    cost = parse(Float64, fields[3]);  # cost of completing the step

    return (source, target, cost)
end;

Build the input path from `CHEME5800_L4D_DATA`, which [`Include.jl`](Include.jl) defines relative to the meeting folder. The resulting `path_to_edge_file::String` does not depend on the directory from which Jupyter was launched:

In [ ]:
path_to_edge_file = joinpath(CHEME5800_L4D_DATA, "Production-Process.edgelist"); # the graph shown in the schematic

[The `MyGraphEdgeModels(...)` constructor](../../../code/src/Files.jl) returns one [`MyGraphEdgeModel` instance](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/types/#VLDataScienceMachineLearningPackage.MyGraphEdgeModel) per parsed step. The `myedgemodels::Dict{Int64, MyGraphEdgeModel}` dictionary keys count data records from zero, so the first parsed step `(1, 2)` has key `0`. These keys record file order; they are not the graph model's final edge ids:

In [ ]:
myedgemodels = MyGraphEdgeModels(path_to_edge_file, edgerecordparser, delim = ',', comment = '#')

[The `build(...)` function](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/factory/#VLDataScienceMachineLearningPackage.build) converts the parsed records into [a `MySimpleDirectedGraphModel` instance](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/types/#VLDataScienceMachineLearningPackage.MySimpleDirectedGraphModel). The resulting `directedgraphmodel::MySimpleDirectedGraphModel` stores the nodes, directed endpoint pairs, step costs, and outgoing-neighbor relationships required by the shortest-path solvers:

In [ ]:
directedgraphmodel = build(MySimpleDirectedGraphModel, myedgemodels);

Before computing a route, inspect the graph model's fields to identify where the builder stored its nodes, costs, adjacency information, and edge ids. [The `typeof(...)` function](https://docs.julialang.org/en/v1/base/base/#Core.typeof) identifies the model's concrete type, and [the `fieldnames(...)` function](https://docs.julialang.org/en/v1/base/base/#Base.fieldnames) lists the fields defined by that type:

In [ ]:
typeof(directedgraphmodel) |> T -> fieldnames(T) # inspect the graph model's stored containers

The `edgesinverse::Dict{Int64, Tuple{Int64, Int64}}` field maps each graph-model edge id to its `(source, target)` pair. Unlike the zero-based parser keys, these ids begin at one and follow the endpoint ordering imposed by the graph builder. The step `(3, 4)`, for example, has parser key `2` but graph-model edge id `4`. The tables and route descriptions below use the graph-model ids:

In [ ]:
directedgraphmodel.edgesinverse

The `edges::Dict{Tuple{Int64, Int64}, Number}` field maps each directed `(source, target)` pair to the cost used during edge relaxation. Verifying that every stored cost is nonnegative matters because Dijkstra's greedy finalization is valid only under that condition. A negative cost would rule out Dijkstra; Bellman–Ford would be the appropriate solver for this acyclic process:

In [ ]:
directedgraphmodel.edges

The next table joins `edgesinverse` with `edges`, placing each graph-model edge id beside its endpoints and cost. This view exposes the exact terms used to calculate the two candidate route costs before running either solver:

In [ ]:
let
    edges = directedgraphmodel.edges;
    edgesinverse = directedgraphmodel.edgesinverse;
    df = DataFrame();
    for i in sort(collect(keys(edgesinverse)))
        (s, t) = edgesinverse[i];
        push!(df, (edge = i, s = s, t = t, cost = edges[(s, t)]));
    end
    pretty_table(df)
end

The routes share edge 1 from vertex 1 to vertex 2 and edge 6 from vertex 5 to vertex 9. Those common costs contribute equally to both plans and therefore cannot determine which route is cheaper. The upper branch uses edges 2, 4, and 5, whose costs total $4+8+2=14$. The lower branch uses edges 3, 7, 8, and 9, whose costs total $2+2+2+2=8$. The six-unit difference between the branches predicts that the lower route will remain cheaper after the shared edges are included.

___

## Task 2: Compute the least-cost route
In this task, we calculate the two candidate route costs and use the smaller value as a benchmark for Dijkstra's algorithm. We then run Bellman–Ford, whose different relaxation schedule provides a second algorithmic check. Hand calculation is practical here because the graph has only two start-to-finish routes.

Let $\langle v_0,v_1,\ldots,v_k\rangle$ denote a route containing $k$ directed steps, where $v_i$ is the process state at position $i$ and every consecutive pair $(v_i,v_{i+1})$ is an edge in the graph. Let $w(v_i,v_{i+1})$ denote the cost of that edge. The total route cost $C$ is defined by:
$$
C = \sum_{i=0}^{k-1} w(v_i, v_{i+1}).
$$

This objective minimizes accumulated cost, not the number of steps. From the edge table, the upper route costs $1+4+8+2+1=16$, whereas the lower route costs $1+2+2+2+2+1=10$. We therefore predict that the six-step lower route is optimal even though the upper route uses only five steps.

[The `L4dProductionPlanning.route_cost(...)` function](src/Compute.jl), which is already complete, evaluates this sum for an ordered vector of vertex ids. It rejects an empty vector and any consecutive vertex pair that is not an edge. Those checks prevent an invalid vertex sequence from being reported as a route cost. The `upper_route::Vector{Int64}` and `lower_route::Vector{Int64}` variables store the two vertex sequences; `upper_cost::Float64` and `lower_cost::Float64` store their computed totals:

In [ ]:
upper_route = [1, 2, 3, 4, 5, 9];    # fewer steps, two of them expensive
lower_route = [1, 2, 6, 7, 8, 5, 9]; # more steps, each of them cheap
upper_cost = L4dProductionPlanning.route_cost(directedgraphmodel.edges, upper_route);
lower_cost = L4dProductionPlanning.route_cost(directedgraphmodel.edges, lower_route);
(upper = upper_cost, lower = lower_cost)

The computed totals reproduce the hand calculation: 16 for the upper route and 10 for the lower route. The lower total establishes the expected Dijkstra distance to vertex 9 and the expected route against which to check the solver.

[The `findshortestpath(...)` function](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/graphs/#VLDataScienceMachineLearningPackage.findshortestpath) accepts the graph model, the node model for the start vertex, and an algorithm. It returns a distance dictionary `d::Dict{Int64, Float64}` and a predecessor dictionary `p::Dict{Int64, Union{Nothing, Int64}}`. The entry `d[v]` is the least cost found from vertex 1 to vertex `v`; `p[v]` records the preceding vertex on that route. The distance answers how much the route costs, while the predecessor chain stores enough information to reconstruct which steps attain that cost. The start vertex has no predecessor:

In [ ]:
(d, p) = let
    startnode = directedgraphmodel.nodes[start_vertex]; # the node model, not the id
    (d, p) = findshortestpath(directedgraphmodel, startnode, algorithm = DijkstraAlgorithm());
    (d, p)
end;

[The `L4dProductionPlanning.reconstruct_route(...)` function](src/Compute.jl) converts the predecessor map into an ordered production route. Starting at vertex 9, it follows predecessors until it reaches vertex 1, whose predecessor is `nothing`, and then reverses the collected vertices. The resulting `shortest_route::Vector{Int64}` identifies the steps that attain `d[9]`:

In [ ]:
shortest_route = L4dProductionPlanning.reconstruct_route(p, finish_vertex);
(distance = d[finish_vertex], route = shortest_route)

The Dijkstra output identifies route $1\rightarrow2\rightarrow6\rightarrow7\rightarrow8\rightarrow5\rightarrow9$ with cost 10. The figure provides a structural check by highlighting those six edges in red while retaining every alternative edge and cost:

In [ ]:
plotroute(directedgraphmodel, shortest_route, node_coordinates)

Dijkstra finalizes the nearest unsettled vertex, whereas Bellman–Ford repeatedly relaxes every edge. Agreement between these different schedules checks that the selected route is not an artifact of one algorithm. It does not independently verify the accumulated route cost; the hand calculation provides that separate check. The second call changes only the algorithm argument. Its results are stored in `d_bellman::Dict{Int64, Float64}` and `p_bellman::Dict{Int64, Union{Nothing, Int64}}`:

In [ ]:
(d_bellman, p_bellman) = let
    startnode = directedgraphmodel.nodes[start_vertex];
    (d, p) = findshortestpath(directedgraphmodel, startnode, algorithm = BellmanFordAlgorithm());
    (d, p)
end;

The following tests encode the three checks explicitly. The route-cost function must reproduce totals 16 and 10 while rejecting invalid routes. Dijkstra must return the predicted lower route and cost. Bellman–Ford must return the same distance and predecessor-derived route:

In [ ]:
@testset "L4d least-cost route" begin
    # Check the route-cost function against the hand arithmetic.
    @test upper_cost == 16.0
    @test lower_cost == 10.0
    @test L4dProductionPlanning.route_cost(directedgraphmodel.edges, [start_vertex]) == 0.0
    @test_throws ArgumentError L4dProductionPlanning.route_cost(directedgraphmodel.edges, Int64[])
    @test_throws ArgumentError L4dProductionPlanning.route_cost(directedgraphmodel.edges, [1, 3])

    # Check Dijkstra's result against the hand calculation.
    @test d[finish_vertex] == lower_cost
    @test shortest_route == lower_route

    # Cross-check Dijkstra's result with Bellman–Ford.
    @test d_bellman[finish_vertex] == d[finish_vertex]
    @test L4dProductionPlanning.reconstruct_route(p_bellman, finish_vertex) == shortest_route
end;

The hand calculation, Dijkstra, and Bellman–Ford all identify the lower route at a cost of 10. The upper route uses one fewer edge, but its route-specific steps cost 14 rather than 8. This comparison confirms that the objective minimizes total weight rather than step count. The six-unit gap also tells us how much cost the upper route must shed before it can become preferable.

___

## Task 3: Evaluate an equipment discount
In this task, we lower the cost of edge `(3, 4)`, recompute the least-cost route, and derive the price at which the selected plan changes. Task 2 established costs of 16 for the upper route and 10 for the lower route. Because edge `(3, 4)` belongs only to the upper route, its cost must fall by more than 6 units before that route becomes strictly cheaper.

Changing `directedgraphmodel` itself would erase the baseline needed to measure the effect of the discount. [The `deepcopy(...)` function](https://docs.julialang.org/en/v1/base/base/#Base.deepcopy) creates an independent graph and independent dictionaries. We store that scenario in `discounted_graphmodel::MySimpleDirectedGraphModel` while retaining the original model for comparison:

In [ ]:
discounted_graphmodel = deepcopy(directedgraphmodel); # a copy we can edit; the baseline stays as it was

[The `MySimpleDirectedGraphModel` type](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/types/#VLDataScienceMachineLearningPackage.MySimpleDirectedGraphModel) stores edge costs in its mutable `edges` dictionary. Assigning a new value to key `(3, 4)` therefore changes the weight used by the next solver call. The `discount_factor` is the fraction of the baseline cost that remains: `0.0` makes the step free, whereas `1.0` leaves its cost unchanged. Apply the zero-cost scenario to the copied model:

In [ ]:
let
    discount_factor = 0.0;           # fraction of the original price that remains: 0.0 is free, 1.0 is no discount
    discounted_steps = [(3, 4)];     # the (source, target) pairs that go on sale
    for step in discounted_steps
        discounted_graphmodel.edges[step] *= discount_factor;
    end
end;

The comparison table verifies that the scenario changed only its intended input. Edge `(3, 4)` should have a discounted cost of 0 and a change $\Delta=-8$; every other edge should have $\Delta=0$:

In [ ]:
let
    edges = directedgraphmodel.edges;
    discounted_edges = discounted_graphmodel.edges;
    edgesinverse = directedgraphmodel.edgesinverse;
    df = DataFrame();
    for i in sort(collect(keys(edgesinverse)))
        (s, t) = edgesinverse[i];
        push!(df, (edge = i, s = s, t = t, cost = edges[(s, t)],
            discounted_cost = discounted_edges[(s, t)], Δ = discounted_edges[(s, t)] - edges[(s, t)]));
    end
    pretty_table(df)
end

The baseline distances and predecessors no longer describe the discounted weights, so we run Dijkstra again on `discounted_graphmodel`. The new distance and predecessor maps are stored in `d₁::Dict{Int64, Float64}` and `p₁::Dict{Int64, Union{Nothing, Int64}}`. The `discounted_route::Vector{Int64}` variable stores the route reconstructed from `p₁`. The start vertex and graph topology remain fixed, isolating the effect of the one cost change:

In [ ]:
(d₁, p₁, discounted_route) = let
    startnode = discounted_graphmodel.nodes[start_vertex];
    (d, p) = findshortestpath(discounted_graphmodel, startnode, algorithm = DijkstraAlgorithm());
    (d, p, L4dProductionPlanning.reconstruct_route(p, finish_vertex))
end;

Plot the recomputed route on the fixed layout. A route switch should appear as a red upper branch containing edge `(3, 4)`, whose label should now be zero:

In [ ]:
plotroute(discounted_graphmodel, discounted_route, node_coordinates)

Setting edge `(3, 4)` to zero lowers the upper-route cost from 16 to 8. The lower-route cost remains 10 because that route does not contain the discounted edge. Dijkstra therefore switches the plan to the upper route. The corresponding edge in `directedgraphmodel` still costs 8, confirming that the baseline was preserved.

The zero-cost scenario proves that a sufficiently large discount can change the plan, but it does not locate the decision boundary. A break-even calculation gives the new step cost at which the two routes tie.

Let $w$ denote the new cost of edge `(3, 4)`. Let $C_{\text{upper}}=16$ and $C_{\text{lower}}=10$ denote the baseline route costs, and let $w_{34}=8$ denote the edge's baseline cost. Replacing $w_{34}$ by $w$ changes the upper-route cost to $C_{\text{upper}}-w_{34}+w$. The lower-route cost remains $C_{\text{lower}}$ because the edge is absent from that route. Setting the two route costs equal and solving for $w$ defines the break-even cost by:
$$
w^{\star} = C_{\text{lower}} - \left(C_{\text{upper}} - w_{34}\right).
$$

For this process, $w^{\star}=10-(16-8)=2$. The routes tie when the step costs 2; the upper route is strictly cheaper below 2, and the lower route is strictly cheaper above 2. The derivation assumes that the discounted step appears exactly once on the candidate route and never on the reference route. If either condition fails, the route costs do not change in the one-for-one pattern represented by the formula.

Your implementation task is to encode this calculation in [the `breakeven_weight(...)` function](src/Compute.jl) in [`src/Compute.jl`](src/Compute.jl). The contract below makes the mathematical assumptions executable by requiring the function to reject routes that place the discounted step incorrectly.

> __What must the break-even function return?__
>
> __Inputs__
>
> * `edges::AbstractDict`: the baseline `(source, target) => cost` dictionary.
> * `candidate_route::AbstractVector{<:Integer}`: the route that contains the discounted step exactly once.
> * `reference_route::AbstractVector{<:Integer}`: the route it competes against, which must not contain the step.
> * `step::Tuple{<:Integer, <:Integer}`: the `(source, target)` pair of the discounted step.
>
> __Output__
>
> * `Float64`: the cost of `step` at which the two routes cost the same. A negative value means no nonnegative price makes the candidate route cheaper.
>
> __Errors__
>
> * [`ArgumentError`](https://docs.julialang.org/en/v1/base/base/#Core.ArgumentError): the step does not appear exactly once on the candidate route, the step appears on the reference route, or either route fails the route-cost checks.

Complete the two TODOs in [the `breakeven_weight(...)` function](src/Compute.jl), save the file, and rerun the setup cell so the notebook loads the revised module:

1. Check that the step appears exactly once on the candidate route and never on the reference route, raising an error otherwise.
2. Compute both route costs with [the `L4dProductionPlanning.route_cost(...)` function](src/Compute.jl) and return the break-even cost from the formula.

The next cell evaluates your implementation for discounted step `(3, 4)` and stores the threshold in `breakeven_cost::Float64`. The final test set verifies both the numerical result and the route-membership checks:

In [ ]:
breakeven_cost = L4dProductionPlanning.breakeven_weight(directedgraphmodel.edges, upper_route, lower_route, (3, 4))

The break-even formula gives a new step cost of 2, exactly 75 percent below the baseline cost of 8. At that price the routes tie; the upper route does not become strictly cheaper until the price falls below 2. The next table evaluates prices of 1.5 and 2.5 on separate baseline copies to check the route selected on each side of the threshold:

In [ ]:
let
    df = DataFrame();
    for price in (breakeven_cost - 0.5, breakeven_cost + 0.5)
        trial = deepcopy(directedgraphmodel);
        trial.edges[(3, 4)] = price;
        (d, p) = findshortestpath(trial, trial.nodes[start_vertex], algorithm = DijkstraAlgorithm());
        route = L4dProductionPlanning.reconstruct_route(p, finish_vertex);
        push!(df, (price = price, distance = d[finish_vertex], route = join(route, " → ")));
    end
    pretty_table(df)
end

The final tests verify both the scenario and the function contract. They confirm that the copied edge cost is zero while the baseline edge cost remains 8. They also confirm that the free-step scenario selects the upper route at cost 8. Finally, they require the break-even implementation to return 2 and enforce its route-membership assumptions:

In [ ]:
@testset "L4d discount scenario" begin
    # Check that the copy changed and the baseline did not.
    @test discounted_graphmodel.edges[(3, 4)] == 0.0
    @test directedgraphmodel.edges[(3, 4)] == 8.0

    # Check that the plan moves to the upper route.
    @test d₁[finish_vertex] == 8.0
    @test discounted_route == upper_route

    # Check the break-even contract.
    @test breakeven_cost == 2.0
    @test_throws ArgumentError L4dProductionPlanning.breakeven_weight(directedgraphmodel.edges, upper_route, lower_route, (6, 7))
    @test_throws ArgumentError L4dProductionPlanning.breakeven_weight(directedgraphmodel.edges, upper_route, lower_route, (1, 2))
end;

The trial at 1.5 selects the upper route, whereas the trial at 2.5 retains the lower route. Together with equality at 2, these results give a complete decision rule: edge `(3, 4)` must cost less than 2, a reduction greater than 75 percent, for the upper route to be strictly cheaper. This threshold holds while every other edge cost remains fixed; simultaneous cost changes require a new comparison.

___

## Summary
The production graph contains an upper route with baseline cost 16 and a lower route with baseline cost 10. Hand calculation, Dijkstra, and Bellman–Ford all select the lower route. Reducing the cost of edge `(3, 4)` below 2 lowers the upper route beneath 10 and reverses that decision.

> __Key Takeaways:__
>
> * **The graph representation defines the decision:** Vertices encode process states, directed edges encode permitted steps, and weights encode step costs. The nonnegative weights justify Dijkstra's algorithm for this process.
> * **Three checks support the selected route:** Hand arithmetic establishes candidate costs of 16 and 10. Dijkstra selects the lower route. Bellman–Ford verifies that result with a different relaxation schedule. The predecessor map recovers the actual production steps.
> * **The break-even cost converts sensitivity into a decision rule:** Preserving the baseline on a separate graph copy isolates the equipment discount. The upper route is strictly cheaper only when edge `(3, 4)` costs less than 2, which requires a reduction greater than 75 percent.

Week 5 keeps the network representation but changes the objective. A maximum-flow model asks how much material the network can carry rather than which route has the smallest accumulated cost, leading from path algorithms to linear programming.

___